# Model Architecture - ChatKasir
- Nama: Achmad Rif'an
- Bagian: AI-1 (Model Architect)

## 1. Imports Library

In [1]:
import os
import re
import json
import sklearn
import pandas as pd
import numpy as np
import tokenizers
import tensorflow as tf
from google.colab import drive
from tokenizers import Tokenizer
from tokenizers.models import WordPiece
from tokenizers.trainers import WordPieceTrainer
from tokenizers.pre_tokenizers import Whitespace
from sklearn.model_selection import train_test_split
from tensorflow.keras.layers import Input, Embedding, Dense, Dropout, LayerNormalization, MultiHeadAttention, Bidirectional, LSTM
from tensorflow.keras.models import Model

# verifikasi versi
print(f"Pandas       : {pd.__version__}")
print(f"NumPy        : {np.__version__}")
print(f"TensorFlow   : {tf.__version__}")
print(f"Scikit-Learn : {sklearn.__version__}")
print(f"Tokenizers   : {tokenizers.__version__}")

# check GPU yang tersedia
print(f"GPU tersedia: {len(tf.config.list_physical_devices('GPU')) > 0}")

Pandas       : 2.2.2
NumPy        : 2.0.2
TensorFlow   : 2.20.0
Scikit-Learn : 1.6.1
Tokenizers   : 0.22.2
GPU tersedia: False


## 2. Data Loading

In [2]:
# Mount Google Drive
drive.mount('/content/drive')

# Definisikan Base Path
BASE_PATH = "/content/drive/MyDrive/ChatKasir"
DATA_PATH = f"{BASE_PATH}/assets/data"
TOKENIZER_PATH = f"{BASE_PATH}/assets/tokenizers"

# URL Dataset dari DS-1
url_food = "https://drive.google.com/uc?id=1xpoFjqAT9K0uwzSpVADm_EfKqG7dxVUI"
url_slang = "https://drive.google.com/uc?id=1G14C1qcqOp06Xs1HFiorE3Us_LLtaBs7"
url_sintetis = "https://drive.google.com/uc?id=1nSqwPqullS6LhfVzh2xp1lYpW5Uw53pG"

# Fungsi membaca CSV dari Google Drive
# def load_gdrive_csv(url):
#    return pd.read_csv(url)

# Memuat ke dalam DataFrame
# df_food = load_gdrive_csv(url_food)
# df_slang = load_gdrive_csv(url_slang)
# df_sintetis = load_gdrive_csv(url_sintetis)
df_food = pd.read_csv(url_food)
df_slang = pd.read_csv(url_slang)
df_sintetis = pd.read_csv(url_sintetis)

print(f"Total data chat sintetis: {len(df_sintetis)} baris")
print(f"Total daftar menu: {len(df_food)} baris")

Mounted at /content/drive
Total data chat sintetis: 100000 baris
Total daftar menu: 18558 baris


In [3]:
# Tampilkan 5 baris pertama data food dan chat sintetis
display(df_food.head())
display(df_sintetis.head())

,name
0,abon
1,abon ayam
2,abon burger
3,abon cheese burger
4,abon goreng ayam


,input_text,product,quantity,price_satuan,pattern
0,pesan nasi goreng babi setengah porsi dan kail...,nasi goreng babi & kailan daging ayam,setengah & segelas,-1,4
1,pesen nasgor sei ayam nya tiga piring porsi [S...,nasgor sei ayam,tiga piring,74000,3
2,setengah guri coffee latte pedes mampus ya kak...,guri coffee latte,setengah,21000,2
3,bu mau pesen 6 boba pedes mampus dong [SEP] ok...,boba,6,39000,2
4,mbak ayam penyet ekonomis dua sama es cendol 5...,ayam penyet ekonomis & es cendol,dua & 5,-1,4


## 3. Tokenizer (WordPiece)

In [4]:
# Gabungkan teks chat sintetis dan nama makanan
semua_teks = df_sintetis['input_text'].astype(str).tolist() + df_food['name'].astype(str).tolist()

# Inisialisasi Tokenizer WordPiece dengan token [UNK] untuk kata tak dikenal
tokenizer = Tokenizer(WordPiece(unk_token="[UNK]"))
tokenizer.pre_tokenizer = Whitespace()  # Pisahkan kata berdasarkan spasi

# Atur Trainer: vocab 5000 untuk domain UMKM
# Special tokens:
# [PAD]: padding
# [UNK]: kata tak dikenal
# [SEP]: pemisah chat pembeli dan penjual
trainer = WordPieceTrainer(
    vocab_size=5000,
    special_tokens=["[PAD]", "[UNK]", "[SEP]"]
)

# Latih Tokenizer dengan teks gabungan
tokenizer.train_from_iterator(semua_teks, trainer)

# Simpan tokenizer ke file JSON
# os.makedirs("..\\assets\\tokenizers", exist_ok=True)
# tokenizer.save("..\\assets\\tokenizers\\tokenizer.json")
# print("Tokenizer disimpan di '..\\assets\\tokenizers\\tokenizer.json'")
os.makedirs(TOKENIZER_PATH, exist_ok=True)
tokenizer.save(f"{TOKENIZER_PATH}/tokenizer.json")
vocab_size = tokenizer.get_vocab_size()
print(f"Tokenizer disimpan. Vocab Size: {vocab_size}")

# Ambil ukuran kosakata akhir
vocab_size = tokenizer.get_vocab_size()
print(f"Ukuran Vocab: {vocab_size}")

# Tes kemampuan tokenizer menangani typo
tes_kalimat = "kk mau pesen 10 daebak ken chicken wings [SEP] baik kak daebak ken chicken wings rp3rb satuan totalnya rp30rb"
hasil_tes = tokenizer.encode(tes_kalimat)

print(f"\n===HASIL TES TOKENIZER===")
print(f"Kalimat asli: {tes_kalimat}")
print(f"Dipecah menjadi tokens: {hasil_tes.tokens}")
print(f"Diubah ke ID angka: {hasil_tes.ids}")


Tokenizer disimpan. Vocab Size: 5000
Ukuran Vocab: 5000

===HASIL TES TOKENIZER===
Kalimat asli: kk mau pesen 10 daebak ken chicken wings [SEP] baik kak daebak ken chicken wings rp3rb satuan totalnya rp30rb
Dipecah menjadi tokens: ['kk', 'mau', 'pesen', '10', 'da', '##ebak', 'ken', 'chicken', 'wings', '[SEP]', 'baik', 'kak', 'da', '##ebak', 'ken', 'chicken', 'wings', 'rp3rb', 'satuan', 'totalnya', 'rp30rb']
Diubah ke ID angka: [340, 153, 181, 211, 2427, 4985, 2997, 157, 447, 2, 182, 90, 2427, 4985, 2997, 157, 447, 1213, 236, 124, 1197]


## 4. Menghitung Max Length

In [5]:
# Hitung panjang token dari setiap baris di dataset chat sintetis
panjang_semua_teks = [len(tokenizer.encode(teks).ids) for teks in df_sintetis['input_text'].astype(str)]

# Cari yang paling panjang
max_length_data = max(panjang_semua_teks)
print(f"Panjang kalimat maksimal asli di dataset: {max_length_data}")

# Membulatkan max_length ke 128
optimal_max_length = 128
print(f"Max Length optimal yang akan digunakan: {optimal_max_length}")

Panjang kalimat maksimal asli di dataset: 51
Max Length optimal yang akan digunakan: 128


## 5. Dataset Preparation

In [6]:
# 1. DEFINISI KAMUS TAG
# Menggunakan format BIO (Begin, Inside, Outside) untuk 3 entitas utama: Produk, Kuantitas, dan Harga.
tag2id = {
    "O": 0,        # Kata biasa
    "B-PROD": 1,   # Kata pertama dari nama produk
    "I-PROD": 2,   # Kata lanjutan dari nama produk
    "B-QTY": 3,    # Kata/angka pertama dari jumlah pesanan
    "I-QTY": 4,    # Kata lanjutan dari jumlah pesanan, contoh: "porsi" pada "2 porsi"
    "B-PRICE": 5,  # Kata/angka pertama dari harga
    "I-PRICE": 6   # Kata lanjutan dari harga, contoh: "ribu" pada "15 ribu"
}

# Mengubah ID angka (0-6) kembali menjadi teks tag (O, B-PROD, dll)
id2tag = {id: tag for tag, id in tag2id.items()}

# Menghitung total jumlah kelas (7 kelas)
num_tags = len(tag2id)
print(f"Jumlah Tag (NUM_TAGS) saat ini: {num_tags}")

# 2. FUNGSI PEMBUAT DATASET NER
def siapkan_data_latih(df, tokenizer, max_len):
    """
    Fungsi ini bertugas mengubah data teks mentah dari CSV menjadi matriks angka
    yang siap dibaca oleh model Transformer (Input IDs dan Label Tags).
    """
    X_input_ids = [] # List untuk menyimpan matriks token kalimat (Fitur)
    Y_ner_tags = []  # List untuk menyimpan matriks tag BIO (Target Label)

    # Memproses setiap baris percakapan yang ada di dalam dataset
    for index, row in df.iterrows():
        teks_chat = str(row['input_text']) # Mengambil kalimat utuh (pembeli + kasir)

        # PENANGANAN MULTI-ITEM
        # Jika kolom produk berisi "nasi goreng & es teh", pisahkan menjadi list: ["nasi goreng", "es teh"]
        daftar_produk = [p.strip() for p in str(row['product']).split("&") if p.strip()]

        # Pecah angka kuantitas (misal "2 & 3") jika ada tanda '&'
        raw_qty = str(row['quantity'])
        daftar_qty = [q.strip() for q in raw_qty.split("&")] if raw_qty != "nan" else []

        # TOKENISASI
        # Ubah teks_chat menjadi deretan ID Token
        encoded_teks = tokenizer.encode(teks_chat).ids

        # Secara default, asumsikan semua kata di kalimat tersebut adalah kata biasa ('O')
        tags = [tag2id['O']] * len(encoded_teks)

        # PENCARI DAN PENEMPEL TAG
        def tag_entity(entity_text, tag_b, tag_i):
            """
            Mencari posisi suatu entitas (misal: "nasi goreng") di dalam kalimat penuh,
            lalu menempelkan tag awalan (B-) dan tag lanjutan (I-) pada posisinya.
            """
            # Abaikan jika entitas kosong, 'nan', atau ditandai -1
            if not entity_text or str(entity_text) == "nan" or str(entity_text) == "-1": return

            # Ubah kata yang dicari menjadi ID token agar bisa dicocokkan
            encoded_ent = tokenizer.encode(str(entity_text)).ids
            panjang_ent = len(encoded_ent)
            if panjang_ent == 0: return

            # Pemindaian (sliding window) sepanjang kalimat
            for i in range(len(encoded_teks) - panjang_ent + 1):
                # Jika urutan token di kalimat sama dengan urutan token entitas yang dicari
                if encoded_teks[i:i+panjang_ent] == encoded_ent:
                    # Pastikan posisi tersebut belum ditag entitas lain (masih 'O')
                    if tags[i] == tag2id['O']:
                        tags[i] = tag2id[tag_b] # tempelkan awalan
                        # Jika entitasnya lebih dari 1 suku kata, tempelkan lanjutannya
                        for j in range(1, panjang_ent):
                            tags[i+j] = tag2id[tag_i] # tempelkan lanjutan
                    break # Berhenti mencari jika sudah ketemu 1 posisi yang cocok

        # 1.TAGGING PRODUK
        # Lakukan perulangan untuk mengecek setiap nama produk yang dipesan
        for nama_produk in daftar_produk:
            tag_entity(nama_produk, 'B-PROD', 'I-PROD')

        # 2. TAGGING KUANTITAS
        for q_str in daftar_qty:
            q_str = str(q_str).strip()
            if q_str and q_str.lower() != "nan":
                # Siapkan list untuk menampung variasi quantity
                variasi_qty = []

                # Kamus alias untuk mendeteksi angka yang dieja dengan huruf
                kamus_angka = {
                    1: ["satu", "sebiji", "seporsi", "sebungkus", "segelas", "sebotol"],
                    2: ["dua", "loro"], 3: ["tiga", "telu"], 4: ["empat", "mpat", "papat"],
                    5: ["lima", "limo"], 6: ["enam", "enem"], 7: ["tujuh", "pitu"],
                    8: ["delapan", "lapan", "wolu"], 9: ["sembilan", "songo"], 10: ["sepuluh", "sepulu"]
                }

                # Handling string dan angka
                try:
                    # 1. Konversi ke angka (untuk mengatasi format '3.0' atau '3')
                    qty_int = int(float(q_str))
                    variasi_qty.append(str(qty_int))

                    # Jika angkanya 1-10, masukkan juga variasi ejaannya ke dalam list
                    if qty_int in kamus_angka:
                        variasi_qty.extend(kamus_angka[qty_int])

                except ValueError:
                    # 2. Jika gagal (ValueError), berarti datanya berupa teks asli
                    # misal: 'setengah', 'seporsi', 'dua'. langsung masukkan ke pencarian
                    variasi_qty.append(q_str.lower())

                # Urutkan dari string terpanjang agar tidak salah potong saat pencarian
                # misal: mencari 'sebungkus' dulu sebelum mencari 'satu'
                variasi_qty.sort(key=len, reverse=True)

                # Cari kemunculan pertama variasi angka tersebut di dalam teks
                for variasi in variasi_qty:
                    # Regex r'\b' untuk memastikan mencari kata utuh
                    pencarian = re.search(r'\b' + re.escape(variasi) + r'\b', teks_chat, re.IGNORECASE)

                    if pencarian:
                        kata_ditemukan = pencarian.group(0)
                        tag_entity(kata_ditemukan, 'B-QTY', 'I-QTY')
                        break # Jika 1 variasi sudah ketemu, berhenti mengecek variasi lain

        # 3. TAGGING HARGA
        # Menggunakan Regex multi-format
        pola_regex_harga = r'\b(?:rp\s*)?\d+(?:rb|k|\s*ribu|\.000)\b'

        # Cari semua kecocokan di dalam seluruh teks chat
        semua_harga_ditemukan = re.findall(pola_regex_harga, teks_chat, re.IGNORECASE)

        if semua_harga_ditemukan:
            # Hapus duplikat (jika angka yang sama disebut 2x) agar lebih efisien
            semua_harga_ditemukan = list(set(semua_harga_ditemukan))

            # Urutkan dari string terpanjang agar tidak salah deteksi
            # misal: memproses 'rp 15.000' lebih dulu sebelum memproses '15.000'
            semua_harga_ditemukan.sort(key=len, reverse=True)

            for harga_teks in semua_harga_ditemukan:
                # Regex r'\b' untuk memastikan mencari kata utuh
                pencarian = re.search(r'\b' + re.escape(harga_teks) + r'\b', teks_chat, re.IGNORECASE)
                if pencarian:
                    harga_ditemukan = pencarian.group(0)
                    tag_entity(harga_ditemukan, 'B-PRICE', 'I-PRICE')

        # PADDING & TRUNCATING
        # Model Deep Learning butuh ukuran matriks yang sama rata
        if len(encoded_teks) > max_len:
            # Jika kalimat aslinya terlalu panjang, truncating
            encoded_teks = encoded_teks[:max_len]
            tags = tags[:max_len]
        else:
            # Jika kalimat aslinya terlalu pendek, tambahkan token kosong di belakangnya (padding)
            selisih = max_len - len(encoded_teks)
            id_pad = tokenizer.token_to_id("[PAD]")

            encoded_teks = encoded_teks + [id_pad] * selisih # isi input dengan token [PAD]
            tags = tags + [tag2id['O']] * selisih            # isi label target dengan tag 'O'

        # Masukkan hasil akhir baris ini ke dalam list matriks utama
        X_input_ids.append(encoded_teks)
        Y_ner_tags.append(tags)

    # Ubah format list Python menjadi NumPy array agar lebih efisien diolah GPU
    return np.array(X_input_ids), np.array(Y_ner_tags)

# 4. EKSEKUSI PEMBUATAN DATASET
# Jalankan fungsi utama dan simpan hasilnya ke variabel X dan Y_ner
X, Y_ner = siapkan_data_latih(df_sintetis, tokenizer, optimal_max_length)

print("\nData sudah siap untuk dilatih")
print(f"Bentuk Input Matriks (X): {X.shape}")
print(f"Bentuk Target Label NER (Y_ner): {Y_ner.shape}")

Jumlah Tag (NUM_TAGS) saat ini: 7

Data sudah siap untuk dilatih
Bentuk Input Matriks (X): (100000, 128)
Bentuk Target Label NER (Y_ner): (100000, 128)


## 6. Split Dataset (Training, Validation, Testing)

In [7]:
# Pisahkan data Training dulu (80%), sisa 20% simpan di variabel sementara (temp)
X_train, X_temp, Y_ner_train, Y_ner_temp = train_test_split(
    X, Y_ner, test_size=0.20, random_state=42
)

# Bagi sisa 20% data sama rata (50-50) untuk Validation dan Testing
X_val, X_test, Y_ner_val, Y_ner_test = train_test_split(
    X_temp, Y_ner_temp, test_size=0.50, random_state=42
)

print("Hasil Pembagian Data:")
print(f"1. Training (80%): {len(X_train)} baris")
print(f"2. Validation (10%): {len(X_val)} baris")
print(f"3. Testing (10%): {len(X_test)} baris")

# Simpan semua array ke dalam satu file kompresi numpy (.npz)
# os.makedirs("..\\assets\\data", exist_ok=True)
# lokasi_simpan = "..\\assets\\data\\dataset_chatkasir.npz"

os.makedirs(DATA_PATH, exist_ok=True)
lokasi_simpan = f"{DATA_PATH}/dataset_chatkasir.npz"

# Simpan X dan Y_ner
np.savez(lokasi_simpan,
         X_train=X_train, Y_ner_train=Y_ner_train,
         X_val=X_val, Y_ner_val=Y_ner_val,
         X_test=X_test, Y_ner_test=Y_ner_test)

print(f"\nData NER berhasil disimpan di: {lokasi_simpan}")

Hasil Pembagian Data:
1. Training (80%): 80000 baris
2. Validation (10%): 10000 baris
3. Testing (10%): 10000 baris

Data NER berhasil disimpan di: /content/drive/MyDrive/ChatKasir/assets/data/dataset_chatkasir.npz


## 7. Arsitektur Transformer

In [8]:
# Arsitektur Transformer + BiLSTM
class TransformerEncoder(tf.keras.layers.Layer):
    def __init__(self, embed_dim, num_heads, ff_dim, rate=0.1, **kwargs):
        super(TransformerEncoder, self).__init__(**kwargs)

        # mengabaikan padding (token [PAD]) saat memproses data
        self.supports_masking = True

        # Multi-Head Attention: melihat hubungan antar kata
        # misal: melihat bahwa kata "2" berhubungan erat dengan kata "porsi"
        self.att = MultiHeadAttention(num_heads=num_heads, key_dim=embed_dim)

        # Feed Forward Network: untuk memperdalam pemahaman makna kata
        self.ffn = tf.keras.Sequential([Dense(ff_dim, activation="relu"), Dense(embed_dim)])

        # Layer Normalization: penyeimbang agar bobot tidak meledak saat AI belajar
        self.layernorm1 = LayerNormalization(epsilon=1e-6)
        self.layernorm2 = LayerNormalization(epsilon=1e-6)

        # Dropout: mencegah AI overfitting dengan mematikan sekian % saraf secara acak
        self.dropout1 = Dropout(rate)
        self.dropout2 = Dropout(rate)

    # Fungsi call() alur perjalanan data (teks) dari awal masuk hingga keluar layer
    def call(self, inputs, training=False, mask=None):
        # Membuat mask agar AI mengabaikan token [PAD] yang kosong
        padding_mask = tf.cast(mask[:, tf.newaxis, :], dtype=tf.int32) if mask is not None else None

        # Teks masuk ke sistem Attention (AI mulai mengaitkan konteks antar kata)
        attn_output = self.att(inputs, inputs, attention_mask=padding_mask)
        attn_output = self.dropout1(attn_output, training=training)

        # Residual Connection 1: menggabungkan pemahaman awal (inputs) dengan konteks baru (attn_output)
        out1 = self.layernorm1(inputs + attn_output)

        # Memasukkan hasil gabungan ke Feed Forward untuk diproses lebih lanjut
        ffn_output = self.ffn(out1)
        ffn_output = self.dropout2(ffn_output, training=training)

        # Residual Connection 2: menggabungkan hasil out1 dengan hasil ffn_output, lalu distabilkan
        return self.layernorm2(out1 + ffn_output)

# Bangun arsitektur dengan Transformer
def build_model(vocab_size, max_length, num_tags):
    # Parameter dasar Transformer
    embed_dim = 128  # Dimensi embedding (representasi kata)
    num_heads = 4    # Jumlah head attention
    ff_dim = 256     # Dimensi feed forward

    # Input Layer berupa ID token
    inputs = Input(shape=(max_length,), name="input_ids")

    # Hidden layer
    x = Embedding(input_dim=vocab_size, output_dim=embed_dim, mask_zero=True)(inputs)  # Ubah ID jadi vektor
    x = TransformerEncoder(embed_dim, num_heads, ff_dim)(x)  # Masukkan ke Transformer Encoder
    x = TransformerEncoder(embed_dim, num_heads, ff_dim)(x)

    # BiLSTM (return_sequences=True agar tidak memadat menjadi 1 angka)
    lstm_output = Bidirectional(LSTM(64, return_sequences=True))(x)

    # Head NER Tunggal (Unified Output Head)
    # Layer Dense berukuran 7 (num_tags) dengan Softmax
    ner_output = Dense(num_tags, activation='softmax', name="ner_output")(lstm_output)

    # Output hanya terdiri dari 1 matriks
    return Model(inputs=inputs, outputs=ner_output)

# Merancang model dengan 7 tag untuk product, quantity, price_satuan
model = build_model(
    vocab_size=vocab_size,
    max_length=optimal_max_length,
    num_tags=num_tags
)

# Ringkasan arsitektur model
model.summary()

Model: "functional_2"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_ids           │ (None, 128)       │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ embedding           │ (None, 128, 128)  │    640,000 │ input_ids[0][0]   │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ not_equal           │ (None, 128)       │          0 │ input_ids[0][0]   │
│ (NotEqual)          │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ transformer_encoder │ (None, 128, 128)  │    330,240 │ embedding[0][0],  │
│ (TransformerEncode… │                   │            │ not_equal[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ transformer_encode… │ (None, 128, 128)  │    330,240 │ transformer_enco… │
│ (TransformerEncode… │                   │            │ not_equal[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ bidirectional       │ (None, 128, 128)  │     98,816 │ transformer_enco… │
│ (Bidirectional)     │                   │            │ not_equal[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ ner_output (Dense)  │ (None, 128, 7)    │        903 │ bidirectional[0]… │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 1,400,199 (5.34 MB)

 Trainable params: 1,400,199 (5.34 MB)

 Non-trainable params: 0 (0.00 B)

## 8. Menyimpan Konfigurasi

In [9]:
config_model = {
    "vocab_size": vocab_size,
    "max_length": optimal_max_length,
    "num_tags": num_tags,
    "embed_dim": 128,
    "num_heads": 4,
    "ff_dim": 256
}

# lokasi_config = "..\\assets\\data\\model_config.json"
lokasi_config = f"{DATA_PATH}/model_config.json"
with open(lokasi_config, "w") as f:
    json.dump(config_model, f, indent=4)

print(f"Konfigurasi arsitektur berhasil disimpan di: {lokasi_config}")

Konfigurasi arsitektur berhasil disimpan di: /content/drive/MyDrive/ChatKasir/assets/data/model_config.json
